# Synthetic Fusion: Chapter 3 reproduction

This notebook runs the complete study workflow from the residential source Parquet, Smart* Home A files and TON-IoT processed streams. It creates new generators, synthetic tables, quality assessments, selected columns, task and baseline results, figures, and a versioned comparison with the reported findings.

The source modules are included in this repository. This notebook explains the sequence and calls the same runner as `run.py`, avoiding a second implementation of the experiments.

**Reference result:** the author run of 27 September 2026, its evaluated numeric-count CTGAN baselines, and the 228-comparison PASS returned on 29 September 2026. This notebook combines generation, the recorded baseline configurations and the same fixed result comparison in one workflow. Each execution produces its own comparison report.

Read [the implementation guide](../docs/IMPLEMENTATION.md), [input guide](../docs/DATA.md), and [reproducibility guide](../docs/REPRODUCIBILITY.md).

## 1. Computational scope

The generation starts from the canonical 1,277,694-row residential Parquet. It does not recreate the earlier packet capture or feature-extraction process. Q4 coupling pairs, their TRAIN/VAL statistics, and the author feature-role annotations are included fixed research inputs. Their identity and internal consistency are checked. The original upstream pair-discovery program was not recovered.

The six router/Zigbee/driver window rows use the documented reconstruction included in `src/window_reconstruction.py`. These boundaries are explicit parts of the supplied workflow.

All model candidates are fitted inside a new run. Candidate arrays produced during that run are normal intermediate outputs; no previous run's continuous candidates or fitted models are imported. Numerical reference files are accessed only for comparisons.

In [ ]:
from pathlib import Path
import json, os, sys
# Works when Jupyter starts in the repository root or its notebooks directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/workflow.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the extracted repository folder.")
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from configuration import load_config
from workflow import new_run, preflight, run_stage, validate_resume, summary, STAGES
from collect_run import collect
print("Repository:", ROOT)
print("Python:", sys.executable)

## 2. Configure input locations and output

Copy `config/example.json` to `config/local.json` and set the residential Parquet, Smart* root, TON-IoT root and output folder. See [the data guide](../docs/DATA.md). On Falcon keep the repository in Home and output in scratch. Other reviewers should use their own absolute paths.

For a fresh run leave `SF_RESUME_RUN` unset. To resume a completed-stage run, set it before starting the kernel or use the terminal route in [the recovery guide](../docs/RECOVERY.md). Do not edit notebook source to resume an existing run.


In [ ]:
config_path = ROOT / "config/local.json"
CONFIG = load_config(config_path if config_path.is_file() else None)
RESUME_RUN = os.environ.get("SF_RESUME_RUN", "").strip()
if RESUME_RUN:
    RUN = Path(RESUME_RUN).expanduser().resolve()
    CONFIG = validate_resume(RUN)
else:
    RUN = new_run(CONFIG)
print(json.dumps(CONFIG, indent=2))
print("SAVE THIS RUN DIRECTORY:", RUN)

## 3. Preflight

Preflight checks the recorded Python and library versions, GPU/runtime, source Parquet SHA-256 and shape, all required public input files, package integrity and 52 coupling-manifest checks. A different environment is reported rather than silently accepted. No training begins until preflight passes.

Use the original Python 3.11.11 environment and the recorded NVIDIA L40S/CUDA 12.8/cuDNN 91002 setup for this numerical reproduction attempt. See `environment/` for the observed versions and hardware.

In [ ]:
checks = preflight(RUN)
failed = [r for r in checks if r["status"] != "PASS"]
print(json.dumps({"status": "BLOCKED" if failed else "PASS", "failed_checks": failed}, indent=2))
if failed:
    print("DIAGNOSTIC_ARCHIVE:", collect(RUN))
    raise RuntimeError("Preflight is blocked. Resolve the reported items before training.")

In [ ]:
def execute(stage):
    try:
        result = run_stage(RUN, stage, resume=bool(RESUME_RUN))
        print(json.dumps(result, indent=2))
        return result
    except BaseException:
        summary(RUN)
        print("DIAGNOSTIC_ARCHIVE:", collect(RUN))
        raise

print("Execution order:", " → ".join(STAGES))

## 4. Controlled activity experiments

Recompute the controlled binary-process experiments using the recorded settings and seeds. Compare pooled-rate, individual-rate and simultaneous-activity rules. This isolates decision behaviour without fitting residential generators.

Primary result: `controlled_stage/controlled/decision_by_setting.csv`.

In [ ]:
result_controlled = execute("controlled")

## 5. Smart* Home A

Parse the six Home A input folders, align one-minute windows, apply the chronological split, assign roles using TRAIN/VAL, fit candidates, and assess TEST features. Structural zeros and eligibility rules are retained. Expected coverage is 293 assigned features and 291 eligible TEST features.

Primary result: `smartstar/ledgers/test_evidence_q1_q2_q3_ledger.csv`.

In [ ]:
result_smartstar = execute("smartstar")

## 6. TON-IoT physical/network checks

Read all 23 processed network CSVs and the seven processed IoT streams. Rebuild aligned windows, derive physical event drivers, select TRAIN/VAL pairs and assess TEST profiles. Also calculate the uninterrupted clock-window sensitivity check. This application assesses real-period consistency; it is not a claim that a synthetic TON-IoT dataset was generated.

Primary result: `toniot/physical_q4/tables/toniot_transfer_claim_scope_summary_physical_q4.csv`.

In [ ]:
result_toniot = execute("toniot")

## 7. Residential generation and baseline evaluation

Run the ordered residential source modules. They load and split the data, assign feature ownership, fit protocol, continuous, binary, event-driver and observation-mask branches, construct candidates, assemble the full synthetic table, evaluate Q1–Q4, and evaluate baseline models.

TRAIN has 766,616 rows, VAL 255,538 and TEST 255,540. Continuous candidates are built fresh and selected using VAL. Original family routing, random seeds, budgets, clipping, rounding and scope rules are retained.

CTGAN treats protocol counts as numeric and retains genuine binary event indicators as categorical. The corrected scopes have 0/12 and 23/45 categorical fields. Its compact-IoT scope is unchanged. The two amended scopes use the same 20,000-row samples, 20 epochs, batch size 500 and two CPU threads as the evaluated amendment. TabDDPM and TimeGAN retain their recorded settings. This is the longest stage; the previous complete execution exceeded 24 hours.

Primary result: `residential_BINARY_V6/reports/cell17_4_baseline_metric_by_artifact.csv`.

In [ ]:
result_residential = execute("residential")

## 8. Additional result summaries

Compute the recorded post-generation tables and pooled/window reconciliations from this run. This stage does not substitute stored reference values into results.

Primary result: `residential_BINARY_V6/reports/review_fix_outputs/window_vs_pooled_reconciliation.csv`.

In [ ]:
result_postrun = execute("postrun")

## 9. Candidate selection and activity-window checks

Apply Q6 policies to the current outputs. The intermediate candidate contains 46 columns; the final conservative selection contains 23. The activity-window check assesses the driver group separately from individual grades. Report the real-versus-real context without changing the original selection policy.

Primary result: `candidate_schema_check.json`.

In [ ]:
result_selection = execute("selection")

## 10. Feature-role inspection

Inspect the source features in the recorded ambiguity scope and carry the author annotations as explicit human inputs. Automated ownership and semantic annotations answer different questions; the latter are not presented as automatically discovered ground truth.

Primary result: `inspection/`.

In [ ]:
result_inspection = execute("inspection")

## 11. Protocol prediction diagnostic

Rebuild the recorded reference-assisted next-minute design on the final candidate. Fit paired real-trained and synthetic-trained random forests using five fixed learner seeds and one worker; score against the same real evaluation targets. Also score persistence, fitted-target mean and fitted-target median.

The reported median and 95th-percentile normalized MAE gaps are 0.092861 and 0.102925. Meeting those numerical limits does not imply predictive superiority: persistence has lower MAE. This task remains exploratory because its design used the inspected TEST period.

Primary result: `protocol_regression/q5_corrected_scores.csv`.

In [ ]:
result_protocol_regression = execute("protocol_regression")

## 12. Two-window branch analysis

Compute two equal contiguous TEST windows for 18 router, four Zigbee and 26 driver fields. Use finite values separately in each dataset, require at least 100 per feature/window, and average two-sided KS statistics. The implemented reconstruction and its provenance are recorded before computation.

Primary result: `window_reconstruction/window_metrics.csv`.

In [ ]:
result_window_reconstruction = execute("window_reconstruction")

## 13. Compare with the current reported results

Run all 228 registered comparisons from the immutable 29 September reference. Both displayed precision and strict numerical tolerance must match. Also require successful CTGAN scopes and no recorded C2ST fallback. Missing evidence cannot pass.

The integrated code retains the label CTGAN; the adapter maps the reference label CTGAN_numeric_counts_v1 only in the two amended scopes. This changes file resolution, not expected numbers or tolerances.

Primary result: `paper_comparison/SUMMARY.json`.

In [ ]:
result_paper_comparison = execute("paper_comparison")

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(RUN / "paper_comparison/comparison.csv").groupby("status").size().rename("checks").to_frame())
scores = pd.read_csv(RUN / "protocol_regression/q5_corrected_scores.csv")
display(scores.groupby("model")[["mae", "normalized_mae", "r2"]].mean())
baselines = pd.read_csv(RUN / "residential_BINARY_V6/reports/cell17_4_baseline_metric_by_artifact.csv")
display(baselines[["baseline", "scope_id", "metric_status", "fair_scope_penalty_score"]])

## 14. Regenerate numerical figures

Render Figures 2, 3 and S1 from the newly computed outputs and retain plotting data. Figure 1 is an authored workflow diagram, not a generated scientific measurement. The supplied thesis supplement is the checked reference document; it is not automatically rewritten to fit a different new result.

Primary result: `figures/figure_provenance.json`.

In [ ]:
result_figures = execute("figures")

In [ ]:
from IPython.display import display, Image
for name in ["fig02_controlled_decisions.png", "fig03_empirical_quality.png", "figS1_candidate_filtering.png"]:
    display(Image(filename=str(RUN / "figures" / name)))

## 15. Record outputs for a future independent-run comparison

Fingerprint generated tables, candidate vectors and the declared result tables. The inventory handles mixed object columns and metadata-only tables. Execution metadata exclusions are listed explicitly. Completion does not itself establish agreement across two independent runs.

Primary result: `repeatability_inventory.json`.

In [ ]:
result_repeatability_inventory = execute("repeatability_inventory")

## 16. Final result and diagnostic archive

A successful reproduction of registered results requires `all_stages_completed: true` and `paper_reference_status: PASS_FOR_VERSIONED_REFERENCE`. A different numerical result is retained for investigation; the notebook does not tune models, relax tolerances or rewrite references to force a match.

Inspect the diagnostic archive printed below alongside the final report. It excludes large model weights and source datasets. Keep full run outputs in scratch for subsequent verification.

In [ ]:
final = summary(RUN)
print(json.dumps(final, indent=2))
print("DIAGNOSTIC_ARCHIVE:", collect(RUN))
if not final["registered_result_agreement"]:
    raise RuntimeError("Execution or numerical agreement is incomplete. Review the saved reports.")

## 17. Independent repeatability

For this stronger test, perform a second independent fresh run with unchanged inputs, package, configuration values and environment. Do not copy fitted candidates between runs. Compare using `compare_runs.py FIRST_RUN SECOND_RUN --output NEW_COMPARISON_DIRECTORY`. See [the results guide](../docs/RESULTS.md) for interpretation and exclusions.

Keep the source version, input hashes, configuration, environment, final report and diagnostic archive with the experiment record. The supplementary PDF provides the reference methods and results.
